# Table Context + Entities (datasets & metrics) with LightOnOCR + GLiNER2

For every table in `pdfs_test/`:

1. **Context** — `caption` (paired per table) + `mentions` (narrative paragraphs).
2. **Entities** — GLiNER2 multi-pass on **table HTML only** (header + body rows; caption is not fed to GLiNER).
   - Same passes as `table_*_benchmark_gliner.ipynb`.
   - Labels `model` / `dataset` / `metric`; export only `datasets` and `metrics`.
   - Entities whose norm key appears **only in the caption** are dropped (e.g. loss names like MRL).
   - Output deduplicated by `normalize_dataset()` / `normalize_metric()` (+ `h@N` ≡ `hits@N` for metrics).

Output: `pdfs_test/table_context_lightonocr.json`

In [1]:
# If needed on a clean server, uncomment these installs:
# !pip install -q torch transformers pypdfium2 pillow beautifulsoup4 "gliner2>=1.2.5"

import json
import os
import re
import tempfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pypdfium2 as pdfium
import torch
from PIL import Image
from bs4 import BeautifulSoup

print("Imports OK")

Imports OK


In [2]:
# Paths and run config
# PDF_DIR holds the test PDFs whose text we OCR for context extraction.
PDF_DIR = Path("./pdfs_test")
OUTPUT_PATH = Path("pdfs_test/table_context_lightonocr.json")

# Optional cache: page-level OCR text is cached here so re-runs are fast.
OCR_CACHE_DIR = Path("./pdfs_test/ocr_cache")

OCR_MODEL_ID = "lightonai/LightOnOCR-2-1B"
OCR_MAX_NEW_TOKENS = 8192
OCR_TARGET_LONGEST = 1540  # px, per LightOnOCR model card

GLINER2_MODEL_ID = "fastino/gliner2-base-v1"
GLINER2_MIN_SCORE = 0.65
GLINER2_MAX_CHARS = 3000

assert PDF_DIR.exists(), f"Missing folder: {PDF_DIR} (put the test PDFs there)"
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
print(f"PDFs found in {PDF_DIR}: {len(PDF_FILES)}")
for p in PDF_FILES:
    print(" -", p.name)

PDFs found in pdfs_test: 5
 - Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function.pdf
 - Adversarial Contrastive Estimation.pdf
 - Binarized Knowledge Graph Embeddings.pdf
 - HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion.pdf
 - Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text.pdf


In [3]:
# Load LightOnOCR
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

if torch.cuda.is_available():
    ocr_device = "cuda"
    ocr_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    ocr_device = "mps"
    ocr_dtype = torch.float16
else:
    ocr_device = "cpu"
    ocr_dtype = torch.float32

print(f"OCR device: {ocr_device} | dtype: {ocr_dtype}")

ocr_processor = LightOnOcrProcessor.from_pretrained(OCR_MODEL_ID)
ocr_model = LightOnOcrForConditionalGeneration.from_pretrained(
    OCR_MODEL_ID,
    torch_dtype=ocr_dtype,
    attn_implementation="eager",
).to(ocr_device)

print("LightOnOCR model loaded")

OCR device: cuda | dtype: torch.bfloat16


You are using a model of type mistral3 to instantiate a model of type lighton_ocr. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/532 [00:00<?, ?it/s]

LightOnOCR model loaded


In [4]:
# Disable PyTorch JIT/TensorExpr fusion (DeBERTa-v3 backbone used by GLiNER2 can trigger
# an nvrtc JIT path that fails on some CUDA builds). Eager kernels are correct + tiny slowdown.
os.environ.setdefault("PYTORCH_JIT", "0")
os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")
for _name, _args in [
    ("_jit_set_profiling_executor", (False,)),
    ("_jit_set_profiling_mode", (False,)),
    ("_jit_override_can_fuse_on_gpu", (False,)),
    ("_jit_override_can_fuse_on_cpu", (False,)),
    ("_jit_set_texpr_fuser_enabled", (False,)),
    ("_jit_set_nvfuser_enabled", (False,)),
]:
    _fn = getattr(torch._C, _name, None)
    if _fn is not None:
        try:
            _fn(*_args)
        except Exception:
            pass

# Load GLiNER2 (same entity extractor used by the metric/dataset notebooks)
from gliner2 import GLiNER2

GLINER2_LABEL_DESCRIPTIONS = {
    "model": "Name of a model, method, or algorithm (e.g. TransE, ComplEx, RotatE).",
    "dataset": "Name of a benchmark dataset or knowledge-graph corpus (e.g. WN18, FB15k, YAGO).",
    "metric": "Name of an evaluation metric used for reporting performance (e.g. MRR, Hits@10, F1).",
}
GLINER2_LABELS = list(GLINER2_LABEL_DESCRIPTIONS.keys())

gliner2_map_location = "cuda" if torch.cuda.is_available() else "cpu"
gliner2_model = GLiNER2.from_pretrained(GLINER2_MODEL_ID, map_location=gliner2_map_location)
print(f"GLiNER2 model loaded: {GLINER2_MODEL_ID} on {gliner2_map_location}")

[W628 16:19:24.562617410 init.cpp:774] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
GLiNER2 model loaded: fastino/gliner2-base-v1 on cuda


In [5]:
# ---- OCR helpers ----------------------------------------------------------
def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = OCR_TARGET_LONGEST) -> Image.Image:
    page = pdf_doc[page_idx]
    img = page.render(scale=200 / 72).to_pil()
    w, h = img.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    return img.convert("RGB") if img.mode != "RGB" else img


def ocr_page(img: Image.Image, max_new_tokens: int = OCR_MAX_NEW_TOKENS) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    img.save(tmp, format="PNG")
    tmp.close()
    try:
        conv = [{"role": "user", "content": [{"type": "image", "url": tmp.name}]}]
        inputs = ocr_processor.apply_chat_template(
            conv, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        )
        inputs = {
            k: v.to(device=ocr_device, dtype=ocr_dtype) if v.is_floating_point() else v.to(ocr_device)
            for k, v in inputs.items()
        }
        with torch.no_grad():
            out = ocr_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[0, inputs["input_ids"].shape[1]:]
        return ocr_processor.decode(gen, skip_special_tokens=True)
    finally:
        os.unlink(tmp.name)


def ocr_pdf_pages(pdf_path: Path) -> List[str]:
    """OCR every page of a PDF to text. Cached per PDF as <stem>_pages.json."""
    cache_file = OCR_CACHE_DIR / f"{pdf_path.stem}_pages.json"
    if cache_file.exists():
        with cache_file.open(encoding="utf-8") as f:
            return json.load(f)["pages"]

    pdf_doc = pdfium.PdfDocument(str(pdf_path))
    pages = []
    try:
        for page_idx in range(len(pdf_doc)):
            print(f"  OCR page {page_idx + 1}/{len(pdf_doc)}", end="\r", flush=True)
            pages.append(ocr_page(render_pdf_page(pdf_doc, page_idx)))
    finally:
        pdf_doc.close()
    print()

    with cache_file.open("w", encoding="utf-8") as f:
        json.dump({"file_name": str(pdf_path), "pages": pages}, f, ensure_ascii=False)
    return pages


# ---- Context patterns -----------------------------------------------------
# A table number: arabic (1, 2.1), roman (IV) or single-letter-prefixed (A1).
_NUM = r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+"

TABLE_BLOCK_RE = re.compile(r"<table\b[^>]*>.*?</table>", re.DOTALL | re.IGNORECASE)
# Caption: "Table N:" / "Table N." / "**Table N**:" at the start of a line/segment.
CAPTION_RE = re.compile(rf"(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+({_NUM})(?:\*\*)?\s*[:.\u2014-]\s+", re.IGNORECASE)
# In-text reference (single or grouped: "Tables 2 and 3").
TABLE_REF_RE = re.compile(rf"\b(?:Table|Tab\.?|TABLE|Tables|TABLES)\s+({_NUM})(?:\s*(?:,|and|&)\s*({_NUM}))*", re.IGNORECASE)
# Strip the leading 'Table(s)' keyword before reading the referenced numbers.
_REF_KEYWORD_RE = re.compile(r"^(?:Tables?|Tab\.?|TABLES?)\s*", re.IGNORECASE)
# Number extractor (no IGNORECASE: avoids matching the lowercase 'l' in 'Table' as roman L).
_REF_NUM_RE = re.compile(r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+")


def normalize_table_number(raw: str) -> str:
    return raw.strip().upper().replace(" ", "")


def _collapse_ws(text: str) -> str:
    return re.sub(r"[ \t]+", " ", text).strip()


def find_page_captions(page_text: str) -> List[dict]:
    """All caption segments on a page, sorted by start position."""
    captions = []
    for m in CAPTION_RE.finditer(page_text):
        start = m.start()
        rest = page_text[start:]
        para = re.split(r"\n\s*\n", rest, maxsplit=1)[0]
        caption = _collapse_ws(para)
        if not caption:
            continue
        captions.append({
            "table_num": normalize_table_number(m.group(1)),
            "caption": caption,
            "start": start,
            "end": start + len(para),
        })
    return captions


def pair_captions_to_tables(page_text: str) -> List[dict]:
    """Assign one caption to each <table> block on the page.

    When caption and table counts match, pair by document order (fixes multi-table pages
    where table N's caption sits before the block but table N+1's caption sits after it).
    Otherwise fall back to the nearest unused caption.
    """
    blocks = list(TABLE_BLOCK_RE.finditer(page_text))
    captions = find_page_captions(page_text)
    if not blocks:
        return []

    results: List[dict] = []
    if len(captions) == len(blocks):
        caps_sorted = sorted(captions, key=lambda c: c["start"])
        for bm, cap in zip(blocks, caps_sorted):
            results.append({
                "match": bm,
                "caption": cap["caption"],
                "table_num": cap["table_num"],
            })
        return results

    used: set = set()
    for bm in blocks:
        t_start, t_end = bm.start(), bm.end()
        best_idx = None
        best_dist = float("inf")
        for ci, cap in enumerate(captions):
            if ci in used:
                continue
            if cap["end"] <= t_start:
                dist = t_start - cap["end"]
            elif cap["start"] >= t_end:
                dist = cap["start"] - t_end
            else:
                dist = 0
            if dist < best_dist:
                best_dist = dist
                best_idx = ci
        if best_idx is not None:
            used.add(best_idx)
            cap = captions[best_idx]
            results.append({
                "match": bm,
                "caption": cap["caption"],
                "table_num": cap["table_num"],
            })
        else:
            results.append({"match": bm, "caption": "", "table_num": None})
    return results


def page_paragraphs_without_tables(page_text: str) -> List[str]:
    text = TABLE_BLOCK_RE.sub(" ", page_text)
    return [_collapse_ws(p) for p in re.split(r"\n\s*\n+", text) if p.strip()]


def find_mentions(pages: List[str]) -> Dict[str, List[dict]]:
    """Map normalized table number -> list of {page, text} narrative paragraphs."""
    mentions: Dict[str, List[dict]] = {}
    for page_idx, page_text in enumerate(pages, start=1):
        for para in page_paragraphs_without_tables(page_text):
            # skip paragraphs that are themselves a caption
            if CAPTION_RE.match(para):
                continue
            nums = set()
            for m in TABLE_REF_RE.finditer(para):
                body = _REF_KEYWORD_RE.sub("", m.group(0))
                for g in _REF_NUM_RE.findall(body):
                    nums.add(normalize_table_number(g))
            for n in nums:
                mentions.setdefault(n, []).append({"page": page_idx, "text": para})
    return mentions


# ---- Table HTML helpers ---------------------------------------------------
def header_and_rows_from_html(html: str) -> Tuple[List[str], List[str]]:
    """Split HTML table into (header_lines, body_lines)."""
    soup = BeautifulSoup(html, "html.parser")
    thead = soup.find("thead")
    tbody = soup.find("tbody")

    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None

    header_lines: List[str] = []
    body_lines: List[str] = []

    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)

    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for tr in trs:
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        if all(c.name == "th" for c in cells) and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)

    if not header_lines and body_lines:
        header_lines = [body_lines[0]]
        body_lines = body_lines[1:]
    return header_lines, body_lines


def table_header_text(html: str) -> str:
    """Header/column lines only — used for entity extraction (avoids model rows)."""
    return "\n".join(header_and_rows_from_html(html)[0])


def table_rows_text(html: str) -> str:
    """Flatten an HTML table into 'cell | cell' lines (header + body)."""
    header_lines, body_lines = header_and_rows_from_html(html)
    return "\n".join(header_lines + body_lines)


print("OCR + context helpers loaded")

OCR + context helpers loaded


In [6]:
# ---- Normalization + GLiNER2 (benchmark string rules only) -------------------
import unicodedata


def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents(s)
    return re.sub(r"\s+", " ", s)


def normalize_dataset(text: object) -> str:
    s = normalize_text(text)
    return s.replace(" ", "").replace("_", "").replace("-", "")


def _strip_trailing_plural(s: str) -> str:
    if not s or "@" in s or not s.isalpha():
        return s
    if len(s) > 3 and s.endswith("s") and not s.endswith("ss"):
        return s[:-1]
    return s


def normalize_metric(text: object) -> str:
    s = normalize_text(text)
    if not s:
        return ""
    s = re.sub(r"[^a-z0-9@]+", "", s)
    return _strip_trailing_plural(s)


def metric_dedup_key(raw: str) -> str:
    """Dedup key for metrics; h@N and Hits@N share the same key."""
    nm = normalize_metric(raw)
    if not nm:
        return ""
    m = re.match(r"^h@(\d+)$", nm)
    if m:
        return f"hits@{m.group(1)}"
    return nm


def _pick_display(candidates: List[str]) -> str:
    """When several raw strings share a dedup key, keep the most informative one."""
    return max(candidates, key=lambda s: (len(s), any(c.isupper() for c in s)))


def _dedup_by_key(items: List[str], key_fn) -> List[str]:
    buckets: Dict[str, List[str]] = {}
    for value in items:
        key = key_fn(value)
        if not key:
            continue
        buckets.setdefault(key, []).append(value)
    return sorted(_pick_display(vals) for vals in buckets.values())


def _clean_entity(value: str) -> str:
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    s = re.sub(r"\\(?:textbf|textit|text|mathbf|mathrm|mathit)\{([^{}]*)\}", r"\1", s)
    for _ in range(3):
        s = re.sub(r"[_^]\{([^{}]*)\}", r"\1", s)
    s = s.replace("{", "").replace("}", "")
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    return s


def _extract_entities_raw(text: str) -> Dict[str, Tuple[str, float]]:
    """GLiNER2 over one text chunk; {entity_text: (label, confidence)}."""
    best: Dict[str, Tuple[str, float]] = {}
    if not text or not text.strip():
        return best
    try:
        result = gliner2_model.extract_entities(
            text[:GLINER2_MAX_CHARS],
            GLINER2_LABEL_DESCRIPTIONS,
            include_confidence=True,
        )
    except Exception as e:
        print(f"  [gliner2 error] {type(e).__name__}: {e}")
        return best

    for label, items in (result or {}).get("entities", {}).items():
        if label not in GLINER2_LABELS:
            continue
        for item in items or []:
            if isinstance(item, dict):
                raw, score = str(item.get("text", "")), float(item.get("confidence", 1.0) or 1.0)
            else:
                raw, score = str(item), 1.0
            value = _clean_entity(raw)
            if score < GLINER2_MIN_SCORE or len(value) < 2:
                continue
            if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
                continue
            prev = best.get(value)
            if prev is None or score > prev[1]:
                best[value] = (label, score)
    return best


def _merge_best(target: Dict[str, Tuple[str, float]], other: Dict[str, Tuple[str, float]]) -> None:
    for value, (label, score) in other.items():
        prev = target.get(value)
        if prev is None or score > prev[1]:
            target[value] = (label, score)


def _keys_from_best(best: Dict[str, Tuple[str, float]], label: str, key_fn) -> set:
    keys = set()
    for value, (lbl, _) in best.items():
        if lbl != label:
            continue
        k = key_fn(value)
        if k:
            keys.add(k)
    return keys


def extract_table_entities(caption: str, html: str) -> Dict[str, List[str]]:
    """GLiNER on table HTML only (header + body rows). Caption is not sent to GLiNER."""
    header_lines, body_lines = header_and_rows_from_html(html)
    header_context = "\n".join(header_lines)

    table_best: Dict[str, Tuple[str, float]] = {}

    if header_context:
        _merge_best(table_best, _extract_entities_raw(header_context))

    for row_text in body_lines:
        prompt = (
            f"Table column headers: {header_context}\nRow: {row_text}"
            if header_context else row_text
        )
        _merge_best(table_best, _extract_entities_raw(prompt))

    if not table_best:
        full = "\n".join(header_lines + body_lines)
        _merge_best(table_best, _extract_entities_raw(full))

    # Caption-only keys (e.g. MRL loss name) — excluded from output.
    caption_only_ds: set = set()
    caption_only_mt: set = set()
    if caption:
        cap_best = _extract_entities_raw(caption)
        cap_ds = _keys_from_best(cap_best, "dataset", normalize_dataset)
        cap_mt = _keys_from_best(cap_best, "metric", metric_dedup_key)
        table_ds = _keys_from_best(table_best, "dataset", normalize_dataset)
        table_mt = _keys_from_best(table_best, "metric", metric_dedup_key)
        caption_only_ds = cap_ds - table_ds
        caption_only_mt = cap_mt - table_mt

    filtered_datasets: List[str] = []
    filtered_metrics: List[str] = []
    for value, (label, _) in table_best.items():
        if label == "model":
            continue
        if label == "dataset":
            k = normalize_dataset(value)
            if k and k not in caption_only_ds:
                filtered_datasets.append(value)
        elif label == "metric":
            k = metric_dedup_key(value)
            if k and k not in {"hits@", "h@"} and k not in caption_only_mt:
                filtered_metrics.append(value)

    return {
        "dataset": _dedup_by_key(filtered_datasets, normalize_dataset),
        "metric": _dedup_by_key(filtered_metrics, metric_dedup_key),
    }


print("GLiNER2 entity helpers loaded")

GLiNER2 entity helpers loaded


In [7]:
def extract_context_for_pdf(pdf_path: Path) -> dict:
    print("\n" + "=" * 70)
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)

    pages = ocr_pdf_pages(pdf_path)
    mentions_by_num = find_mentions(pages)

    tables = []
    for page_idx, page_text in enumerate(pages, start=1):
        paired = pair_captions_to_tables(page_text)
        for t_i, item in enumerate(paired, start=1):
            html = item["match"].group(0)
            caption = item["caption"] or ""
            table_num = item["table_num"]
            mentions = mentions_by_num.get(table_num, []) if table_num else []

            ents = extract_table_entities(caption, html)

            tables.append({
                "table_id": f"{pdf_path.stem}_p{page_idx}_t{t_i}",
                "table_label": f"Table {table_num}" if table_num else None,
                "page": page_idx,
                "caption": caption,
                "mentions": mentions,
                "datasets": ents["dataset"],
                "metrics": ents["metric"],
            })

    print(f"  tables={len(tables)} | "
          f"with_caption={sum(1 for t in tables if t['caption'])} | "
          f"datasets={sum(len(t['datasets']) for t in tables)} | "
          f"metrics={sum(len(t['metrics']) for t in tables)}")

    return {
        "paper": pdf_path.stem,
        "num_tables": len(tables),
        "tables": tables,
    }


print("Pipeline loaded")

Pipeline loaded


In [8]:
# Run on every PDF in pdfs_test and export the final JSON
documents = []
for pdf_path in PDF_FILES:
    documents.append(extract_context_for_pdf(pdf_path))

result_json = {
    "description": "Per-table context (caption + mentions) with datasets & metrics via GLiNER2 (same multi-pass extraction as table_*_benchmark_gliner notebooks).",
    "pdf_dir": str(PDF_DIR.resolve()),
    "num_documents": len(documents),
    "total_tables": sum(d["num_tables"] for d in documents),
    "documents": documents,
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(result_json, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"Saved: {OUTPUT_PATH}")
print(f"Documents: {result_json['num_documents']} | Tables: {result_json['total_tables']}")


Processing: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function.pdf
  tables=2 | with_caption=2 | datasets=4 | metrics=5

Processing: Adversarial Contrastive Estimation.pdf
  tables=4 | with_caption=4 | datasets=1 | metrics=5

Processing: Binarized Knowledge Graph Embeddings.pdf
  tables=5 | with_caption=5 | datasets=10 | metrics=6

Processing: HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion.pdf
  tables=4 | with_caption=4 | datasets=6 | metrics=5

Processing: Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text.pdf
  tables=2 | with_caption=2 | datasets=0 | metrics=1

DONE
Saved: pdfs_test/table_context_lightonocr.json
Documents: 5 | Tables: 17


In [9]:
# Quick preview
with OUTPUT_PATH.open(encoding="utf-8") as f:
    preview = json.load(f)

print(f"{preview['description']}")
print(f"num_documents: {preview['num_documents']} | total_tables: {preview['total_tables']}")

for doc in preview["documents"]:
    print("\n" + "-" * 60)
    print(f"Paper: {doc['paper']}  (tables: {doc['num_tables']})")
    for t in doc["tables"]:
        print(f"  {t['table_id']} (page {t['page']}) | mentions={len(t['mentions'])} "
              f"| datasets={t['datasets']} | metrics={t['metrics']}")
        if t["caption"]:
            print(f"    caption: {t['caption'][:140]}")

Per-table context (caption + mentions) with datasets & metrics via GLiNER2 (same multi-pass extraction as table_*_benchmark_gliner notebooks).
num_documents: 5 | total_tables: 17

------------------------------------------------------------
Paper: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function  (tables: 2)
  Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function_p5_t1 (page 5) | mentions=2 | datasets=['FB15k', 'WN18'] | metrics=['Hits@10(%)', 'Mean', 'filter', 'raw']
    caption: Table 1: Link prediction results. Comparison of models implemented with loss function of MRL, Limited-base loss, and adaptive margin loss co
  Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function_p5_t2 (page 5) | mentions=1 | datasets=['FB15k', 'wn18'] | metrics=['Learning rate']
    caption: Table 2: Optimal Setting. Representation of different setting considering hyperparame